#1 GIAI NÉN DỮ LIỆU VÀ KIỂM TRA DUỜNG DẪN DATA

In [ ]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.9/46.9 kB 390.3 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.0/96.0 kB 5.7 MB/s eta 0:00:00


In [ ]:
import os
import sys
from pathlib import Path
import yaml
import torch
from ultralytics import YOLO
import zipfile


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# # 2. Hàm chuyên dụng cho file .zip
# def extract_zip_only(zip_path: str, extract_to: str) -> None:
#     """
#     Chỉ nhận và giải nén file .zip vào Google Drive.
#     """
#     # Kiểm tra tồn tại
#     if not os.path.exists(zip_path):
#         raise FileNotFoundError(f"Không tìm thấy file tại: {zip_path}")

#     # Chặn nếu không phải đuôi .zip
#     if not zip_path.lower().endswith('.zip'):
#         raise ValueError(f"Lỗi: Hàm chỉ nhận file định dạng .zip! File truyền vào: {zip_path}")

#     # Kiểm tra file có đúng định dạng zip hợp lệ hay bị hỏng không
#     if not zipfile.is_zipfile(zip_path):
#         raise zipfile.BadZipFile(f"Lỗi: File '{zip_path}' không phải file zip hợp lệ hoặc bị lỗi tải về dở dang.")

#     # Tạo thư mục đích nếu chưa có
#     os.makedirs(extract_to, exist_ok=True)

#     print(f"Bắt đầu giải nén: {os.path.basename(zip_path)}")
#     print(f"vị trí trên Drive: {extract_to}")

#     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#         # Kiểm tra tính toàn vẹn của các file bên trong (CRC checksum)
#         corrupt_file = zip_ref.testzip()
#         if corrupt_file:
#             raise zipfile.BadZipFile(f"File zip bị hỏng dữ liệu tại: {corrupt_file}")

#         zip_ref.extractall(extract_to)

#     print("Giải nén thành công!")

In [ ]:
# # Có thể thay đổi link path theo đường dẫn trên drive của bạn

# zip_path = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/rotate_bee640.yolo26.zip"
# extract_to = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION"
# extract_zip_only(zip_path,extract_to)

### **Trước khi train bạn hãy chỉnh lại đường dẫn trong file yaml sao cho đúng với đường dẫn trên máy bạn**
```python
#(bạn có thể đổi đường dẫn này dựa trên máy của mình)
path: /content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION  
train: train/images
val: valid/images
test: test/images

nc: 3
names: ['bee', 'bee ', 'hornet']
```


In [ ]:
yaml_path= r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/data.yaml"
with open(yaml_path, 'r') as f:
  data = yaml.safe_load(f)
  for key, value in data.items():
    print(f"{key}: {value}")

path: /content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION
train: train/images
val: valid/images
test: test/images
nc: 3
names: ['bee', 'bee ', 'hornet']


In [ ]:
def check_paths(yaml_file):
  with open(yaml_file, 'r', encoding='utf-8') as f:
      data = yaml.safe_load(f)
      # lấy đường dẫn gốc
      if 'path' in data and data['path']:
          root = data['path']
      else:
        root = os.path.dirname(os.path.abspath(yaml_file))

  for key in ["train", "test", "val", "valid"]:
    if key in data and data[key]:
      sub_path = data[key]
      # kiểm tra đường dẫn của train, test, valid có phải là tuyệt đối hay tượng đối
      # nếu là tuyệt đối thì giữ nguyên, không thì ghép với root
      if os.path.isabs(sub_path):
        full_path = sub_path
      else:
        full_path = os.path.normpath(os.path.join(root, sub_path))

      print(f"[PASS] {key}: {full_path}" if os.path.exists(full_path) else f"[ERROR] {key}: {sub_path}")
# Chạy kiểm tra:
yaml_file_path = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/data.yaml"
check_paths(yaml_file_path)

[PASS] train: /content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/train/images
[PASS] test: /content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/test/images
[PASS] val: /content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/valid/images


#2. kIỂM TRA THIẾT BỊ TRAIN
nếu không phải GPU thì sẽ báo lỗi không thực hiện việc train
- yêu cầu bật chết độ GPU trên google colab

In [ ]:
# Kiểm tra GPU Colab
if not torch.cuda.is_available():
    sys.exit("[ERROR]  Colab đang ở chế độ CPU! Hãy vào Runtime -> Change runtime type -> Chọn GPU (T4)")

device = 0
print(f"[PASS] Đang sử dụng GPU: {torch.cuda.get_device_name(0)}")

[PASS] Đang sử dụng GPU: Tesla T4


#3. TRAIN MODEL YOLO26 BẢN NANO VÀ SMALL

In [ ]:
# Bản Yolo26 small

ans = input("Y/N to continue: ")
if ans.lower() == "y":
    # Khởi tạo model (Dùng bản chính thức yolov8n.pt hoặc yolov11n.pt)
    model = YOLO(r"yolo26s.pt")  # hoặc đường dẫn tuyệt đối đến file

    # 2. Định nghĩa chuỗi đường dẫn
    OUTPUT_PATH = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/MODELS"
    # Lưu ý: kiểm tra kỹ tên thư mục trên Drive là 'DECTECTION' hay 'DETECTION'
    YAML_PATH = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/data.yaml"

    # Ghép đường dẫn thư mục lưu kết quả bằng os.path.join
    PROJECT_DIR = os.path.join(OUTPUT_PATH, "bee_wasp_detection")

    print("\n--- BẮT ĐẦU HUẤN LUYỆN TRÊN COLAB ---")
    results = model.train(
        data=YAML_PATH,
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=2,              # Giữ ở mức 2 để tránh tràn bộ nhớ chia sẻ (/dev/shm) trên Colab
        project=PROJECT_DIR,
        name="yolo26s_img640_baseline_v1",
        exist_ok=True
    )

    print("\nHuấn luyện hoàn tất!")

else:
  pass


Y/N to continue: N


In [ ]:
# Bản Yolo26 nano

ans = input("Y/N to continue: ")
if ans.lower() == "y":
    # Khởi tạo model (Dùng bản chính thức yolov8n.pt hoặc yolov11n.pt)
    model = YOLO(r"yolo26s.pt")  # hoặc đường dẫn tuyệt đối đến file

    # 2. Định nghĩa chuỗi đường dẫn
    OUTPUT_PATH = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/MODELS"
    # Lưu ý: kiểm tra kỹ tên thư mục trên Drive là 'DECTECTION' hay 'DETECTION'
    YAML_PATH = r"/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/data.yaml"

    # Ghép đường dẫn thư mục lưu kết quả bằng os.path.join
    PROJECT_DIR = os.path.join(OUTPUT_PATH, "bee_wasp_detection")

    print("\n--- BẮT ĐẦU HUẤN LUYỆN TRÊN COLAB ---")
    results = model.train(
        data=YAML_PATH,
        epochs=100,
        imgsz=640,
        batch=16,
        device=0,
        workers=2,              # Giữ ở mức 2 để tránh tràn bộ nhớ chia sẻ (/dev/shm) trên Colab
        project=PROJECT_DIR,
        name="yolo26s_img640_baseline_v1-nano",
        exist_ok=True
    )

    print("\nHuấn luyện hoàn tất!")

else:
  pass

Y/N to continue: y

--- BẮT ĐẦU HUẤN LUYỆN TRÊN COLAB ---
Ultralytics 8.4.173 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/PROJECTS/BEE-SYSTEMS-PROJECT/PROJECT-DATA/BEE-AND-WASP-DECTECTION/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, 